### Imports

In [19]:
import os
import json
import pickle
import uuid

from datetime import datetime

import pandas as pd

### Create Escalation Folder

In [20]:
os.makedirs(
    "../data/escalations",
    exist_ok=True
)

os.makedirs(
    "../logs",
    exist_ok=True
)

### Load Multi-LLM Validation Results

In [21]:
with open(
    "../data/output/multi_llm_judge_results.json",
    "r"
) as f:

    judge_results = json.load(f)

judge_results

{'question': 'What is a savings account?',
 'intent_confidence': 92,
 'relevance_score': 95,
 'groundedness_score': 56.666666666666664,
 'correctness_score': 61.666666666666664,
 'hallucination_score': 1.6666666666666667,
 'consensus_score': 59.0,
 'agreement_score': 93,
 'trust_score': 78.42,
 'trust_level': 'MEDIUM',
 'decision': {'status': 'CAUTION', 'human_review': False},
 'judge_70b': {'groundedness': 90,
  'correctness': 95,
  'hallucination': 5,
  'safety': 98,
  'overall_score': 92,
  'verdict': 'PASS'},
 'judge_8b': {'groundedness': 80,
  'correctness': 90,
  'hallucination': 0,
  'safety': 100,
  'overall_score': 85,
  'verdict': 'PASS'}}

### Extract Judge Metrics

In [22]:
consensus_score = judge_results["consensus_score"]

groundedness_score = judge_results["groundedness_score"]

hallucination_score = judge_results["hallucination_score"]

question = judge_results["question"]

intent_confidence = judge_results["intent_confidence"]

relevance_score = judge_results["relevance_score"]

trust_score = judge_results["trust_score"]

trust_level = judge_results["trust_level"]

decision = judge_results["decision"]

print("Intent Confidence :", intent_confidence)
print("Relevance Score   :", relevance_score)
print("Groundedness      :", groundedness_score)
print("Hallucination     :", hallucination_score)
print("Consensus Score   :", consensus_score)
print("Trust Score       :", trust_score)
print("Trust Level       :", trust_level)

Intent Confidence : 92
Relevance Score   : 95
Groundedness      : 56.666666666666664
Hallucination     : 1.6666666666666667
Consensus Score   : 59.0
Trust Score       : 78.42
Trust Level       : MEDIUM


### Governance Decision Rules

In [23]:
def determine_response_action(
    trust_score
):

    if trust_score >= 90:

        return {
            "status":"APPROVED",
            "show_response":True,
            "human_review":False
        }

    elif trust_score >= 75:

        return {
            "status":"CAUTION",
            "show_response":True,
            "human_review":False
        }

    else:

        return {
            "status":"ESCALATE",
            "show_response":False,
            "human_review":True
        }

### Final Decision

In [24]:
final_decision = determine_response_action(
    trust_score
)

final_decision

{'status': 'CAUTION', 'show_response': True, 'human_review': False}

### Session ID Generator

In [25]:
def generate_session_id():

    timestamp = datetime.now().strftime(
        "%Y%m%d%H%M%S"
    )

    random_part = str(uuid.uuid4())[:6]

    return f"{timestamp}{random_part}"

### Create Escalation Ticket

In [26]:
def create_escalation_ticket(

    question,
    trust_score,
    trust_level

):

    ticket = {

        "session_id":
        generate_session_id(),

        "timestamp":
        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

        "question":
        question,

        "trust_score":
        trust_score,

        "trust_level":
        trust_level,

        "status":
        "OPEN"

    }

    return ticket

### Save Escalation Ticket

In [27]:
def save_ticket(ticket):

    file_name = (

        "../data/escalations/"
        f"{ticket['session_id']}.json"

    )

    with open(
        file_name,
        "w"
    ) as f:

        json.dump(
            ticket,
            f,
            indent=4
        )

    return file_name

#### Auto Escalation

In [28]:
if final_decision["human_review"]:

    ticket = create_escalation_ticket(

        question,
        trust_score,
        trust_level

    )

    ticket_path = save_ticket(ticket)

    print(
        "Escalation Created:"
    )

    print(ticket_path)

else:

    print(
        "Human Review Not Required"
    )

Human Review Not Required


#### Audit Log Creation

In [29]:
audit_log = {

    "session_id":
    generate_session_id(),

    "timestamp":
    datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    ),

    "question":
    question,

    "intent_confidence":
    intent_confidence,

    "relevance_score":
    relevance_score,

    "groundedness_score":
    groundedness_score,

    "hallucination_score":
    hallucination_score,

    "consensus_score":
    consensus_score,

    "trust_score":
    trust_score,

    "trust_level":
    trust_level,

    "decision":
    final_decision["status"]

}

### Save Audit Log

In [30]:
with open(
    "../logs/audit_log.json",
    "w"
) as f:

    json.dump(
        audit_log,
        f,
        indent=4
    )

print(
    "Audit Log Saved Successfully"
)

Audit Log Saved Successfully


### Governance Report

In [31]:
governance_report = {

    "validation_layers": [

        "Intent Classification",

        "Knowledge Base Relevance",

        "Groundedness Validation",

        "Hallucination Detection",

        "Multi LLM Judge"

    ],

    "trust_score":
    trust_score,

    "trust_level":
    trust_level,

    "final_status":
    final_decision["status"]

}

#### Save Governance Report

In [32]:
with open(
    "../logs/governance_report.json",
    "w"
) as f:

    json.dump(
        governance_report,
        f,
        indent=4
    )

print(
    "Governance Report Saved"
)

Governance Report Saved


### Enterprise Governance Pipeline

In [33]:
def run_governance_pipeline():

    return {

        "question":
        question,

        "trust_score":
        trust_score,

        "trust_level":
        trust_level,

        "decision":
        final_decision,

        "audit_log":
        "../logs/audit_log.json",

        "governance_report":
        "../logs/governance_report.json"

    }

### Run Pipeline

In [34]:
result = run_governance_pipeline()

result

{'question': 'What is a savings account?',
 'trust_score': 78.42,
 'trust_level': 'MEDIUM',
 'decision': {'status': 'CAUTION',
  'show_response': True,
  'human_review': False},
 'audit_log': '../logs/audit_log.json',
 'governance_report': '../logs/governance_report.json'}

## Key Insights

1. This notebook is the final governance layer of the Banking AI Assistant.

2. It does not perform validation itself.

3. It consumes validation outputs from:

   - Intent Classification
   - Knowledge Relevance
   - Groundedness Validation
   - Hallucination Detection
   - Multi-LLM Judge

4. Trust Score determines whether:

   - Response is Approved
   - Response is Cautionary
   - Human Escalation is Required

5. All interactions are audit logged.

6. Governance reports are generated for compliance purposes.

7. Low trust responses automatically create escalation tickets for human review.

8. This architecture follows enterprise-grade AI governance principles used in regulated banking environments.